# Pillar 2: Single-drug FAERS severity

Target: serious-outcome reports divided by eligible submitted reports for a drug structure. This is a descriptive FAERS report fraction, not clinical incidence or patient-level toxicity risk. First rebuild and audit a time-separated cohort (train on 2025 Q1–Q4; test on new cases from 2026 Q1–Q2). Only train after the label outputs and manifest pass review. Colab is used for preparation and training.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, os, sys, subprocess

# Find the user's actual Drive data root, regardless of mount alias.
DATA_CANDIDATES = [Path("/content/gdrive_auditddi_transfer/MyDrive/Projects/auditddi"), Path("/content/gdrive_auditddi/MyDrive/Projects/auditddi"), Path("/content/drive/MyDrive/Projects/auditddi")]
DATA = next((p for p in DATA_CANDIDATES if p.is_dir()), None)
if DATA is None:
    from google.colab import drive
    MOUNT = Path("/content/gdrive_auditddi_toxicity")
    if MOUNT.exists() and any(MOUNT.iterdir()): MOUNT = Path("/content/gdrive_auditddi_toxicity2")
    drive.mount(str(MOUNT))
    DATA = MOUNT / "MyDrive/Projects/auditddi"
assert DATA.is_dir(), f"Google Drive project data folder not found: {DATA}"
# Pull the shared GitHub code. A fresh checkout avoids accidentally using an old notebook copy.
REPO = Path("/content/auditddi_single_drug_toxicity")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", "main", "https://github.com/akhilau-git/auditddi.git", str(REPO)], check=True)
else:
    dirty = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip()
    assert not dirty, "This code checkout has local changes. Use a fresh Colab runtime before pulling."
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
RESULTS_BASE = DATA / "results"
os.environ["AUDITDDI_DATA_BASE"] = str(DATA)
os.environ["AUDITDDI_RESULTS_BASE"] = str(RESULTS_BASE)
sys.path.insert(0, str(REPO / "src"))
RUNS = RESULTS_BASE / "pillars" / "single_drug_toxicity"
RUNS.mkdir(parents=True, exist_ok=True)
print("Colab preparation/training environment; code:", REPO)
print("Git commit:", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True).stdout.strip())
print("Drive data:", DATA)
print("Pillar outputs:", RUNS)

In [ ]:
%pip install -r {REPO / "requirements_colab.txt"}

In [ ]:
# Confirm the six quarterly ASCII archives, the drug-name/SMILES bridge, and the builder script.
ARCHIVE_DIR = DATA / "faers/historical_ascii_quarters"
BRIDGE = DATA / "processed/seven_branch/toxicity_bridge_candidate_2026q2.csv"
QUARTER_ARCHIVES = [ARCHIVE_DIR / f"faers_ascii_{q}.zip" for q in ["2025q1", "2025q2", "2025q3", "2025q4", "2026q1"]] + [DATA / "faers/faers_ascii_2026q2.zip"]
BUILDER = REPO / "scripts/build_faers_multiquarter_toxicity_labels.py"
TRAINER = REPO / "scripts/train_single_drug_toxicity_temporal.py"
import py_compile
py_compile.compile(str(BUILDER), doraise=True)
py_compile.compile(str(TRAINER), doraise=True)
required = [BRIDGE, BUILDER, TRAINER, *QUARTER_ARCHIVES]
missing = [str(p) for p in required if not p.is_file()]
assert not missing, "Required multi-quarter toxicity inputs are missing:\n" + "\n".join(missing)
for p in QUARTER_ARCHIVES: print(f"FOUND {p.name}: {p.stat().st_size / 1024**2:.1f} MB")
print("Bridge:", BRIDGE)
print("Builder:", BUILDER)

In [ ]:
# Build structure-level labels: 2025 reports for training; 2026 H1 new cases for temporal testing.
OUT = RUNS / "label_builds" / f"faers_2025_train_2026h1_test_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=True)
command = [sys.executable, "-u", str(BUILDER), "--data-root", str(DATA), "--archive-dir", str(ARCHIVE_DIR), "--bridge", str(BRIDGE), "--output-dir", str(OUT)]
log_path = OUT / "label_build.log"
print("Building multi-quarter FAERS labels. Output:", OUT)
with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process, log_path.open("w", encoding="utf-8") as log:
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    return_code = process.wait()
if return_code != 0: raise RuntimeError(f"Label build failed with code {return_code}. See {log_path}")
MANIFEST = OUT / "cohort_manifest.json"
assert MANIFEST.is_file(), f"Builder did not save its manifest: {MANIFEST}"
summary = json.loads(MANIFEST.read_text(encoding="utf-8"))
print("Label build complete. Manifest:", MANIFEST)
print("Train/test CASEID overlap:", summary["train_test_case_overlap"])
print("Suspect-role structures: train=", summary["train_suspect_structures"], "test=", summary["test_suspect_structures"], "shared=", summary["suspect_train_test_structure_overlap"])

In [ ]:
# Audit the generated cohort before any toxicity model is trained.
import pandas as pd
train_labels = pd.read_csv(OUT / "train_2025_suspect_roles.csv")
test_labels = pd.read_csv(OUT / "test_new_cases_2026h1_suspect_roles.csv")
required_cols = {"canonical_smiles", "serious_outcome_reports", "n_reports", "serious_outcome_recorded_fraction"}
for name, frame in [("train_2025", train_labels), ("test_2026h1", test_labels)]:
    assert required_cols.issubset(frame.columns), f"{name} missing columns: {sorted(required_cols - set(frame.columns))}"
    assert frame["canonical_smiles"].is_unique, f"{name} has duplicate structures"
    assert (frame["n_reports"] >= frame["serious_outcome_reports"]).all()
    assert frame["serious_outcome_recorded_fraction"].between(0, 1).all()
print("TRAIN 2025:", len(train_labels), "structures; reports=", int(train_labels.n_reports.sum()), "mean fraction=", round(train_labels.serious_outcome_recorded_fraction.mean(), 4))
print("TEST 2026 H1:", len(test_labels), "structures; reports=", int(test_labels.n_reports.sum()), "mean fraction=", round(test_labels.serious_outcome_recorded_fraction.mean(), 4))
print("Structures in both periods:", len(set(train_labels.canonical_smiles) & set(test_labels.canonical_smiles)))
print("This audit does not train a model. Review these counts and the cohort manifest before temporal evaluation.")

In [ ]:
# One-seed temporal pilot: 2025 labels fit the models; 2026 H1 remains test-only.
TRAIN_OUT = RUNS / "temporal_models" / OUT.name
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
train_csv = OUT / "train_2025_suspect_roles.csv"
test_csv = OUT / "test_new_cases_2026h1_suspect_roles.csv"
command = [sys.executable, "-u", str(TRAINER), "--train-labels", str(train_csv), "--test-labels", str(test_csv), "--output-dir", str(TRAIN_OUT), "--seeds", "203", "--epochs", "80", "--patience", "10", "--device", "auto"]
train_log = TRAIN_OUT / "temporal_training.log"
print("Starting one-seed temporal model comparison. Log:", train_log)
with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process, train_log.open("w", encoding="utf-8") as log:
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    return_code = process.wait()
if return_code != 0: raise RuntimeError(f"Temporal pilot failed with code {return_code}. See {train_log}")
print("Temporal pilot complete. Metrics:", TRAIN_OUT / "temporal_metrics.csv")